In [26]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

In [27]:
train = pd.read_csv("train_black_friday.csv")
print(train.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 550068 entries, 0 to 550067
Data columns (total 12 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   User_ID                     550068 non-null  int64  
 1   Product_ID                  550068 non-null  object 
 2   Gender                      550068 non-null  object 
 3   Age                         550068 non-null  object 
 4   Occupation                  550068 non-null  int64  
 5   City_Category               550068 non-null  object 
 6   Stay_In_Current_City_Years  550068 non-null  object 
 7   Marital_Status              550068 non-null  int64  
 8   Product_Category_1          550068 non-null  int64  
 9   Product_Category_2          376430 non-null  float64
 10  Product_Category_3          166821 non-null  float64
 11  Purchase                    550068 non-null  int64  
dtypes: float64(2), int64(5), object(5)
memory usage: 50.4+ MB
None


In [28]:
test = pd.read_csv("test_black_friday.csv")
print(test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 233599 entries, 0 to 233598
Data columns (total 11 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   User_ID                     233599 non-null  int64  
 1   Product_ID                  233599 non-null  object 
 2   Gender                      233599 non-null  object 
 3   Age                         233599 non-null  object 
 4   Occupation                  233599 non-null  int64  
 5   City_Category               233599 non-null  object 
 6   Stay_In_Current_City_Years  233599 non-null  object 
 7   Marital_Status              233599 non-null  int64  
 8   Product_Category_1          233599 non-null  int64  
 9   Product_Category_2          161255 non-null  float64
 10  Product_Category_3          71037 non-null   float64
dtypes: float64(2), int64(4), object(5)
memory usage: 19.6+ MB
None


In [29]:
train["Type"] = "train"
test["Type"] = "test"
fullData = pd.concat([train,test],axis=0)
print(fullData.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 783667 entries, 0 to 233598
Data columns (total 13 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   User_ID                     783667 non-null  int64  
 1   Product_ID                  783667 non-null  object 
 2   Gender                      783667 non-null  object 
 3   Age                         783667 non-null  object 
 4   Occupation                  783667 non-null  int64  
 5   City_Category               783667 non-null  object 
 6   Stay_In_Current_City_Years  783667 non-null  object 
 7   Marital_Status              783667 non-null  int64  
 8   Product_Category_1          783667 non-null  int64  
 9   Product_Category_2          537685 non-null  float64
 10  Product_Category_3          237858 non-null  float64
 11  Purchase                    550068 non-null  float64
 12  Type                        783667 non-null  object 
dtypes: float64(3),

In [30]:
fullData["Product_ID"] = fullData["Product_ID"].astype("category").cat.codes
#Changes data type to category and takes each as its own code

In [31]:
IDcol = ["User_ID", "Product_ID"]
flagcol = ["Type"]
target = ["Purchase"]
categoricalcol = ["Gender","Age","City_Category","Stay_In_Current_City_Years"]
numcol = list(set(fullData.columns) - set(IDcol) - set(flagcol) - set(target) - set(categoricalcol))

In [32]:
print(fullData.isnull().sum())
print(fullData[["Product_Category_2","Product_Category_3","Purchase"]].describe())

User_ID                            0
Product_ID                         0
Gender                             0
Age                                0
Occupation                         0
City_Category                      0
Stay_In_Current_City_Years         0
Marital_Status                     0
Product_Category_1                 0
Product_Category_2            245982
Product_Category_3            545809
Purchase                      233599
Type                               0
dtype: int64
       Product_Category_2  Product_Category_3       Purchase
count       537685.000000       237858.000000  550068.000000
mean             9.844506           12.668605    9263.968713
std              5.089093            4.125510    5023.065394
min              2.000000            3.000000      12.000000
25%              5.000000            9.000000    5823.000000
50%              9.000000           14.000000    8047.000000
75%             15.000000           16.000000   12054.000000
max             18

In [33]:
fullData[numcol] = fullData[numcol].fillna(fullData[numcol].mean())
print(fullData.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 783667 entries, 0 to 233598
Data columns (total 13 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   User_ID                     783667 non-null  int64  
 1   Product_ID                  783667 non-null  int16  
 2   Gender                      783667 non-null  object 
 3   Age                         783667 non-null  object 
 4   Occupation                  783667 non-null  int64  
 5   City_Category               783667 non-null  object 
 6   Stay_In_Current_City_Years  783667 non-null  object 
 7   Marital_Status              783667 non-null  int64  
 8   Product_Category_1          783667 non-null  int64  
 9   Product_Category_2          783667 non-null  float64
 10  Product_Category_3          783667 non-null  float64
 11  Purchase                    550068 non-null  float64
 12  Type                        783667 non-null  object 
dtypes: float64(3),

In [34]:
le = LabelEncoder()
for col in categoricalcol:
    fullData[col] = le.fit_transform(fullData[col])
print(fullData.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 783667 entries, 0 to 233598
Data columns (total 13 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   User_ID                     783667 non-null  int64  
 1   Product_ID                  783667 non-null  int16  
 2   Gender                      783667 non-null  int32  
 3   Age                         783667 non-null  int32  
 4   Occupation                  783667 non-null  int64  
 5   City_Category               783667 non-null  int32  
 6   Stay_In_Current_City_Years  783667 non-null  int32  
 7   Marital_Status              783667 non-null  int64  
 8   Product_Category_1          783667 non-null  int64  
 9   Product_Category_2          783667 non-null  float64
 10  Product_Category_3          783667 non-null  float64
 11  Purchase                    550068 non-null  float64
 12  Type                        783667 non-null  object 
dtypes: float64(3),

In [35]:
features = list(set(fullData.columns) - set(flagcol) - set(IDcol) - set(target))
fullData[features] = fullData[features]/fullData[features].max()

In [ ]:
train = fullData[fullData["Type"]=="train"]
test = fullData[fullData["Type"]=="test"]
Xtrain = train[features]
ytrain = train[target]
Xtest = test[features]
ytest = test[target]

In [38]:
DLmodel = Sequential([Dense(12,activation="relu"),
                      Dropout(0.2),
                      Dense(12,activation="relu"),
                      Dropout(0.2),
                      Dense(1,activation="linear")
                      ])
DLmodel.compile(optimizer=Adam(),loss="mse",metrics=["mae"])
earlyStop = EarlyStopping(monitor="val_loss", patience = 3, restore_best_weights = True)
DLmodel.fit(Xtrain,ytrain,epochs=50,batch_size=64,validation_split=0.1,verbose=1,callbacks=[earlyStop])

Epoch 1/50
7736/7736 [==============================] - 5s 629us/step - loss: 42334644.0000 - mae: 5086.1177 - val_loss: 27678202.0000 - val_mae: 4199.9531
Epoch 2/50
7736/7736 [==============================] - 5s 619us/step - loss: 28097896.0000 - mae: 4145.6758 - val_loss: 22690220.0000 - val_mae: 3731.9158
Epoch 3/50
7736/7736 [==============================] - 5s 618us/step - loss: 26181818.0000 - mae: 3937.4509 - val_loss: 20922290.0000 - val_mae: 3516.1187
Epoch 4/50
7736/7736 [==============================] - 5s 619us/step - loss: 24910320.0000 - mae: 3798.7959 - val_loss: 20205928.0000 - val_mae: 3416.4397
Epoch 5/50
7736/7736 [==============================] - 5s 619us/step - loss: 24065258.0000 - mae: 3712.2075 - val_loss: 20049386.0000 - val_mae: 3413.6802
Epoch 6/50
7736/7736 [==============================] - 5s 622us/step - loss: 23686546.0000 - mae: 3679.6169 - val_loss: 20129450.0000 - val_mae: 3413.1147
Epoch 7/50
7736/7736 [==============================] - 5s 620us